# Operaciones de aprendizaje automático
## Maestría en Inteligencia Artificial Aplicada

### TC5061 | Proyecto MiBici: propuesta y base reproducible | Colaborativo

#### **Equipo:** 7
#### **Nombre:** Luis Arturo Sánchez Davila
#### **Matrícula:** A01840576

## Objetivo
Notebook de **exploración** del proyecto MiBici: carga y descripción del dataset de viajes ya consolidado en Parquet, análisis exploratorio de datos (EDA) y evidencia para las decisiones de limpieza que acordará el equipo.

La pregunta que me corresponde es la **P? (TIPO)**: <!-- TODO: pegar la pregunta de negocio asignada -->

La lógica que se reutilice se trasladará al módulo del proyecto; en este notebook se desarrolla y se documenta la evidencia que la justifica.

## Dataset
**Datos abiertos de MiBici** (sistema de bicicleta pública del Área Metropolitana de Guadalajara), en la versión **ya convertida a Parquet** en `0.01-fat-exploracion-inicial`.
- Archivos: `datos_abiertos_AAAA_MM.parquet`, uno por mes, en `data/external/`. El portal publica **CSV**, no Parquet; si los Parquet no existen, el notebook los genera a partir del CSV del portal (ver 1.1).
- Cada registro es un viaje: `viaje_id`, `usuario_id`, `genero`, `anio_de_nacimiento`, `inicio_del_viaje`, `fin_del_viaje`, `origen_id` y `destino_id`.
- Fuente original: [Datos abiertos de MiBici](https://mibici.net/es/datos-abiertos/)
- Rango analizado: **enero de 2020 a septiembre de 2026**, acordado por el equipo para homologar el EDA. El portal ya no publica 2023-04 ni 2023-05 (ver `0.02-dsh-exploracion-carga`).

## Nota de reproducibilidad
El notebook se ejecuta desde la carpeta `notebooks/` con el entorno descrito en el README. Los Parquet de `data/external/` y los CSV de `data/raw/` no se versionan en Git. La primera ejecución descarga los CSV faltantes del portal y genera los Parquet; las siguientes reutilizan los que ya existen.

In [4]:
# Importación de librerías y registro de versiones (evidencia de reproducibilidad)
from pathlib import Path
import re
import sys
import time
from urllib.parse import urljoin

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow
import requests
import seaborn as sns

print("Python      :", sys.version.split()[0])
print("pandas      :", pd.__version__)
print("numpy       :", np.__version__)
print("pyarrow     :", pyarrow.__version__)
print("requests    :", requests.__version__)
print("matplotlib  :", matplotlib.__version__)
print("seaborn     :", sns.__version__)

Python      : 3.11.9
pandas      : 3.0.5
numpy       : 1.26.4
pyarrow     : 25.0.1
requests    : 2.34.2
matplotlib  : 3.11.2
seaborn     : 0.13.2


## Configuración global

Todos los valores que se usan en más de una celda se definen una sola vez aquí:
- **Rutas:** CSV crudos del portal (`data/raw/`), Parquet mensuales (`data/external/`) y datos intermedios (`data/interim/`).
- **Fuente y rango:** URL del portal, meses inicial y final (inclusivos), meses que el portal ya no publica y encabezados HTTP.
- **Random seed:** para que cualquier muestreo sea reproducible.
- **Opciones de visualización:** pandas muestra todas las columnas.
- **Colores:** paleta Okabe-Ito, apta para daltonismo, referida por nombre.

In [5]:
# Rutas del proyecto (el notebook se ejecuta desde notebooks/)
RUTA_PROYECTO = Path.cwd().parent
RUTA_VIAJES_CRUDOS = RUTA_PROYECTO / "data" / "raw" / "viajes"
RUTA_PARQUET = RUTA_PROYECTO / "data" / "external"
RUTA_INTERMEDIOS = RUTA_PROYECTO / "data" / "interim"

# Rango de meses a analizar (inclusivos)
MES_INICIO = "2020-01"
MES_FIN = "2026-09"

# Fuente de los datos
URL_PORTAL = "https://mibici.net/es/datos-abiertos/"

# El portal dejó de publicar estos meses; solo estos huecos se aceptan (ver 0.02-dsh-exploracion-carga)
MESES_SIN_PUBLICAR = ["2023-04", "2023-05"]

# El portal rechaza (error 406) las peticiones con el User-Agent por defecto de requests
ENCABEZADOS_HTTP = {"User-Agent": "Mozilla/5.0 (proyecto academico MLOps TC5061)",
                    "Accept": "text/html,text/csv,*/*",
}

SEMILLA = 42

# Opciones de visualización de pandas
pd.set_option("display.max_columns", None)   # mostrar todas las columnas
pd.set_option("display.max_colwidth", None)  # no recortar textos largos

# Paleta Okabe-Ito (apta para daltonismo)
AZUL = "#0072B2"
NARANJA = "#E69F00"
VERDE = "#009E73"

RUTA_VIAJES_CRUDOS.mkdir(parents=True, exist_ok=True)
RUTA_PARQUET.mkdir(parents=True, exist_ok=True)
RUTA_INTERMEDIOS.mkdir(parents=True, exist_ok=True)

## Paso 1: Carga y descripción del dataset

En este paso se obtienen los Parquet mensuales, se leen y se valida que la carga sea completa antes de describir el dataset. Cualquier diferencia se documenta como hallazgo y se resuelve con evidencia, no con supuestos.


### 1.1 Obtención de los Parquet

El portal solo publica **CSV**. Los Parquet de `data/external/` son una conversión local, así que para cada mes del rango:
1. Si ya existe `datos_abiertos_AAAA_MM.parquet`, se reutiliza.
2. Si no, se toma el CSV de `data/raw/viajes/`; si tampoco existe, se descarga del portal (los enlaces se leen de la página porque cada archivo vive en una carpeta con identificador distinto).
3. El CSV se convierte a Parquet con nombres de columna en minúsculas sin acentos (`anio_de_nacimiento`), fechas como `datetime` y enteros con tipo que admite nulos.

La codificación (UTF-8 o latin-1) y el formato de fecha (`AAAA-MM-DD hh:mm:ss` o `DD/MM/AAAA hh:mm`) varían entre archivos; se detectan por archivo y, si un valor no se convierte, el notebook se detiene en lugar de dejar fechas nulas sin avisar.

In [6]:
# Obtención de los Parquet faltantes: descarga del CSV y conversión
COLUMNAS_PARQUET = ["viaje_id", "usuario_id", "genero", "anio_de_nacimiento",
                    "inicio_del_viaje", "fin_del_viaje", "origen_id", "destino_id"]
TIPOS_ENTEROS = {"viaje_id": "Int64", "usuario_id": "Int64", "anio_de_nacimiento": "Int16",
                 "origen_id": "Int32", "destino_id": "Int32"}
FORMATOS_FECHA = ["%Y-%m-%d %H:%M:%S", "%d/%m/%Y %H:%M"]


def nombre_parquet(periodo):
    """Regresa el nombre del Parquet de un periodo 'AAAA-MM'."""
    return f"datos_abiertos_{periodo.replace('-', '_')}.parquet"


def enlaces_del_portal():
    """Regresa {periodo: URL} de los CSV de viajes publicados en el portal."""
    respuesta = requests.get(URL_PORTAL, headers=ENCABEZADOS_HTTP, timeout=30)
    respuesta.raise_for_status()
    enlaces = {}
    for href in re.findall(r"href=['\"]([^'\"]+)['\"]", respuesta.text):
        coincidencia = re.search(r"datos_abiertos_(\d{4})_(\d{2})\.csv", href)
        if coincidencia:
            enlaces[f"{coincidencia[1]}-{coincidencia[2]}"] = urljoin(URL_PORTAL, href)
    return enlaces


def csv_a_parquet(ruta_csv, ruta_parquet):
    """Convierte un CSV de viajes a Parquet con columnas, fechas y tipos normalizados."""
    contenido = ruta_csv.read_bytes()
    try:
        contenido.decode("utf-8")
        codificacion = "utf-8"
    except UnicodeDecodeError:
        codificacion = "latin-1"  # decodifica cualquier byte; se usa como último recurso

    viajes = pd.read_csv(ruta_csv, encoding=codificacion, dtype=str)
    if viajes.shape[1] != len(COLUMNAS_PARQUET):
        raise ValueError(f"{ruta_csv.name} tiene {viajes.shape[1]} columnas, se esperaban {len(COLUMNAS_PARQUET)}")
    viajes.columns = COLUMNAS_PARQUET  # por posición: el nombre de 'Año_de_nacimiento' llega corrupto en algunos archivos

    for columna in ("inicio_del_viaje", "fin_del_viaje"):
        con_dato = viajes[columna].notna().sum()
        for formato in FORMATOS_FECHA:
            convertida = pd.to_datetime(viajes[columna], format=formato, errors="coerce")
            if convertida.notna().sum() == con_dato:
                viajes[columna] = convertida
                break
        else:
            raise ValueError(f"{ruta_csv.name}: {columna} tiene fechas que no coinciden con {FORMATOS_FECHA}")
    for columna, tipo in TIPOS_ENTEROS.items():
        viajes[columna] = pd.to_numeric(viajes[columna], errors="coerce").astype(tipo)
    viajes["genero"] = viajes["genero"].astype("category")

    viajes.to_parquet(ruta_parquet, index=False)
    return len(viajes)


# Meses del rango cuyo Parquet no existe y que el portal sí debería publicar
periodos = [str(p) for p in pd.period_range(MES_INICIO, MES_FIN, freq="M")]
pendientes = [p for p in periodos
              if not (RUTA_PARQUET / nombre_parquet(p)).exists() and p not in MESES_SIN_PUBLICAR]
print("Meses del rango:", len(periodos), "| Parquet por generar:", len(pendientes))

if pendientes:
    enlaces = enlaces_del_portal()
    sin_enlace = [p for p in pendientes
                  if p not in enlaces and not (RUTA_VIAJES_CRUDOS / f"datos_abiertos_{p.replace('-', '_')}.csv").exists()]
    if sin_enlace:
        raise ValueError(f"Meses sin Parquet, sin CSV y sin enlace en el portal: {sin_enlace}")

    descargados = 0
    for periodo in pendientes:
        ruta_csv = RUTA_VIAJES_CRUDOS / f"datos_abiertos_{periodo.replace('-', '_')}.csv"
        if not ruta_csv.exists():
            respuesta = requests.get(enlaces[periodo], headers=ENCABEZADOS_HTTP, timeout=60)
            respuesta.raise_for_status()
            ruta_csv.write_bytes(respuesta.content)
            descargados += 1
            time.sleep(1)  # pausa entre descargas para no saturar el portal
        registros = csv_a_parquet(ruta_csv, RUTA_PARQUET / nombre_parquet(periodo))
        print(f"{periodo}: {registros} registros")
    print("CSV descargados en esta ejecución:", descargados)

Meses del rango: 81 | Parquet por generar: 39
2023-07: 331787 registros
2023-08: 354810 registros
2023-09: 363859 registros
2023-10: 383239 registros
2023-11: 384464 registros
2023-12: 314164 registros
2024-01: 391591 registros
2024-02: 383732 registros
2024-03: 390513 registros
2024-04: 405308 registros
2024-05: 410806 registros
2024-06: 365000 registros
2024-07: 364177 registros
2024-08: 394894 registros
2024-09: 390072 registros
2024-10: 418690 registros
2024-11: 391089 registros
2024-12: 340049 registros
2025-01: 380184 registros
2025-02: 372529 registros
2025-03: 405110 registros
2025-04: 366193 registros
2025-05: 395891 registros
2025-06: 356611 registros
2025-07: 368186 registros
2025-08: 374727 registros
2025-09: 376789 registros
2025-10: 420064 registros
2025-11: 380328 registros
2025-12: 335420 registros
2026-01: 357180 registros
2026-02: 333756 registros
2026-03: 376243 registros
2026-04: 351851 registros
2026-05: 315895 registros
2026-06: 355303 registros
2026-07: 388257 re

**Observaciones de la obtención:**
- **Origen:** <!-- TODO: cuántos Parquet ya existían y cuántos se generaron desde CSV -->
- **Descargas:** <!-- TODO: cuántos CSV se bajaron del portal y cuántos ya estaban en data/raw -->
- **Meses sin datos:** <!-- TODO: confirmar que solo faltan 2023-04 y 2023-05 -->
- **Limitación:** <!-- TODO: esta conversión replica la de 0.01 (nombres y tipos); verificar con Fernando que el esquema coincide -->

In [8]:
# Lectura y unión de los Parquet mensuales dentro del rango
PATRON_ARCHIVO = re.compile(r"datos_abiertos_(\d{4})_(\d{2})\.parquet$")

archivos = []
for ruta in sorted(RUTA_PARQUET.glob("datos_abiertos_*.parquet")):
    coincidencia = PATRON_ARCHIVO.search(ruta.name)
    periodo = f"{coincidencia.group(1)}-{coincidencia.group(2)}" if coincidencia else None
    if periodo and MES_INICIO <= periodo <= MES_FIN:
        archivos.append((periodo, ruta))

if not archivos:
    raise FileNotFoundError(f"No hay Parquet en {RUTA_PARQUET}; ejecutar antes la celda 1.1")

viajes = pd.concat(
    [pd.read_parquet(ruta).assign(periodo_archivo=periodo) for periodo, ruta in archivos],
    ignore_index=True,
)

print("Archivos leídos:", len(archivos))
print(f"Registros      : {len(viajes):,}")
print("Columnas       :", viajes.shape[1])

Archivos leídos: 79
Registros      : 26,389,162
Columnas       : 9


**Observaciones de la lectura:**
- **Archivos:** <!-- TODO: N archivos leídos de los M meses del rango; indicar huecos -->
- **Registros:** <!-- TODO: total de registros y mes con más / menos viajes -->
- **Contraste con 0.02:** <!-- TODO: comparar el total contra los 26389162 registros del inventario de 0.02 -->

### 1.3 Descripción del dataset

Se revisan los tipos, los nulos y el rango de fechas para saber qué columnas requieren limpieza.

In [9]:
# Columnas, tipos, valores no nulos y memoria
viajes.info(show_counts=True, memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 26389162 entries, 0 to 26389161
Data columns (total 9 columns):
 #   Column              Non-Null Count     Dtype         
---  ------              --------------     -----         
 0   viaje_id            26389162 non-null  Int64         
 1   usuario_id          26389162 non-null  Int64         
 2   genero              26253930 non-null  category      
 3   anio_de_nacimiento  26355366 non-null  Int16         
 4   inicio_del_viaje    26389162 non-null  datetime64[ns]
 5   fin_del_viaje       26389162 non-null  datetime64[ns]
 6   origen_id           26389162 non-null  Int32         
 7   destino_id          26389162 non-null  Int32         
 8   periodo_archivo     26389162 non-null  str           
dtypes: Int16(1), Int32(2), Int64(2), category(1), datetime64[ns](2), str(1)
memory usage: 1.6 GB


In [10]:
# Nulos y valores distintos por columna
resumen = pd.DataFrame({
    "nulos": viajes.isna().sum(),
    "% nulos": (viajes.isna().mean() * 100).round(2),
    "distintos": viajes.nunique(),
})
resumen

,nulos,% nulos,distintos
viaje_id,0,0.00,26388404
usuario_id,0,0.00,81115
genero,135232,0.51,2
anio_de_nacimiento,33796,0.13,87
inicio_del_viaje,0,0.00,22616495
fin_del_viaje,0,0.00,22569727
origen_id,0,0.00,475
destino_id,0,0.00,475
periodo_archivo,0,0.00,79


In [11]:
# Rango de fechas de inicio y fin de los viajes
print("Inicio:", viajes["inicio_del_viaje"].min(), "->", viajes["inicio_del_viaje"].max())
print("Fin   :", viajes["fin_del_viaje"].min(), "->", viajes["fin_del_viaje"].max())

Inicio: 2020-01-01 06:02:20 -> 2026-09-30 23:59:48
Fin   : 2020-01-01 06:05:38 -> 2026-10-01 00:14:52


**Observaciones de la descripción del dataset:**
- **Estructura:** <!-- TODO: registros, columnas y memoria -->
- **Nulos:** <!-- TODO: columnas con nulos y porcentaje -->
- **Identificadores repetidos:** <!-- TODO: viaje_id distintos vs registros -->
- **Fechas:** <!-- TODO: rango de inicio y fin y qué implica -->
- **Limitación:** <!-- TODO: lo que este análisis aún no confirma -->